In [679]:
import polars as pl
import os
from datetime import datetime
import re
from unidecode import unidecode
from BancoDados.rotina_banco import BancoDados
import json
import requests
from math import ceil

# Fonte Dados

In [680]:
os.getcwd()

'C:\\Users\\JoãoPedroResplandesd\\Estudos\\ProjetosPython\\ProjetoIngestao'

In [681]:
CAMINHO_ARQUIVO: str = r"C:\Users\JoãoPedroResplandesd\Estudos\Pos_Graduacao_Engenharia_de_Dados\Preparacao_Orquestacao_Fluxo_de_Dados\Unidade02\Materiais\ProjetoChurn\Fonte\SIS_CRM_TB_CHURN.txt"

In [682]:
with open(CAMINHO_ARQUIVO, 'r', encoding='utf-8') as arquivo:
    dados = arquivo.readlines()

In [752]:
ult_atualizacao_arquivo = datetime.strftime(datetime.fromtimestamp(os.path.getmtime(CAMINHO_ARQUIVO)), '%Y-%m-%d %H:%M:%S')
ult_atualizacao_arquivo

'2026-09-03 22:30:15'

In [753]:
os.path.exists(CAMINHO_ARQUIVO) # Verifica se o arquivo existe

True

# Transformação

In [683]:
conjunto = [
    [valor.strip() for valor in linha.replace('\t', ',').replace('\n', '').split(',')]
    for linha in dados
]

In [684]:
mapeamento = {
    r'\b(masculino|masc|m)\b': 'M',
    r'\b(bigenero|bi|b)\b': 'B',
    r'\b(feminino|fem|fe|f)\b': 'F',
    r'\b(agenero|ag|a)\b': 'A',
    r'\b(outros|outro|o)\b': 'O'
}

In [685]:
def tratar_genero(texto: str) -> str:
    texto = unidecode(texto).lower()
    for padrao, valor in mapeamento.items():
        texto = re.sub(padrao, valor, texto, flags=re.IGNORECASE | re.VERBOSE)
    return texto

In [686]:
dicionario_tabela = {}
for i, coluna in enumerate(conjunto[0]):
    lista_dados = []
    for dado in conjunto[1:]:
        lista_dados.append(dado[i].upper())
    dicionario_tabela[coluna] = lista_dados

In [687]:
dicionario_tabela['cpf'] = [cpf.replace('.', '').replace('-', '') for cpf in dicionario_tabela['cpf']]
dicionario_tabela['idgenero'] = [tratar_genero(texto=genero) for genero in dicionario_tabela['idgenero']]
dicionario_tabela['ID'] = [int(id) for id in dicionario_tabela['ID']]
dicionario_tabela['indcliente'] = [int(indcliente) if indcliente != '' else 0 for indcliente in dicionario_tabela['indcliente']]

In [688]:
tabela = pl.DataFrame(dicionario_tabela)
tabela.columns = [coluna.lower() for coluna in tabela.columns]
tabela

id,cpf,nome,sobrenome,idgenero,estado_civil,idade,uf,escolaridade,indcliente,mes,ano
i64,str,str,str,str,str,str,str,str,i64,str,str
1,"""12345678901""","""ALICE""","""ALVES""","""F""","""3""","""25""","""SE""","""11""",0,"""05""","""2024"""
2,"""23456700112""","""ANA""","""AMARAL""","""F""","""2""","""56""","""CE""","""6""",1,"""05""","""2024"""
3,"""34567800223""","""ALINE""","""ABREU""","""F""","""1""","""38""","""PE""","""3""",1,"""05""","""2024"""
4,"""45678900334""","""AMANDA""","""ANDRADE""","""F""","""6""","""72""","""BA""","""7""",1,"""05""","""2024"""
5,"""56789000445""","""ADRIANA""","""ARAÚJO""","""F""","""4""","""29""","""MA""","""5""",1,"""05""","""2024"""
…,…,…,…,…,…,…,…,…,…,…,…
502,"""34567800223""","""ALINE""","""ABREU""","""F""","""1""","""38""","""PE""","""3""",1,"""05""","""2024"""
503,"""45656719989""","""ÁGATA""","""HAUGEN""","""M""","""3""","""34""","""MA""","""6""",1,"""05""","""2024"""
504,"""46178257774""","""SOLANO""","""BASTOS""","""M""","""1""","""68""","""PI""","""11""",0,"""05""","""2024"""


# Verifica Valores Duplicados

In [689]:
lista_cpfs = tabela['cpf'].value_counts().filter(pl.col('count') > 1)['cpf'].to_list()
lista_cpfs

['34567800223',
 '45656709989',
 '12345678901',
 '45656719989',
 '46178257774',
 '45656729989']

In [690]:
# ids a serem removidos por duplicidade
l = sorted([id for id in tabela.filter(pl.col('cpf').is_in(lista_cpfs)).sort('cpf')['id'].unique().to_list()])
id_duplicatos = l[int(len(l) / 2):]
print(id_duplicatos)

[501, 502, 503, 504, 506, 507]


In [691]:
tabela_dados_unicos = tabela.filter(~pl.col('id').is_in(id_duplicatos))
tabela_dados_unicos

id,cpf,nome,sobrenome,idgenero,estado_civil,idade,uf,escolaridade,indcliente,mes,ano
i64,str,str,str,str,str,str,str,str,i64,str,str
1,"""12345678901""","""ALICE""","""ALVES""","""F""","""3""","""25""","""SE""","""11""",0,"""05""","""2024"""
2,"""23456700112""","""ANA""","""AMARAL""","""F""","""2""","""56""","""CE""","""6""",1,"""05""","""2024"""
3,"""34567800223""","""ALINE""","""ABREU""","""F""","""1""","""38""","""PE""","""3""",1,"""05""","""2024"""
4,"""45678900334""","""AMANDA""","""ANDRADE""","""F""","""6""","""72""","""BA""","""7""",1,"""05""","""2024"""
5,"""56789000445""","""ADRIANA""","""ARAÚJO""","""F""","""4""","""29""","""MA""","""5""",1,"""05""","""2024"""
…,…,…,…,…,…,…,…,…,…,…,…
496,"""57293457370""","""SIMÃO""","""BERNARDES""","""M""","""2""","""46""","""PB""","""6""",0,"""05""","""2024"""
497,"""95417857471""","""SÉRGIO""","""BORGES""","""M""","""3""","""56""","""CE""","""7""",0,"""05""","""2024"""
498,"""81942657572""","""SALOMÃO""","""BITTENCOURT""","""M""","""4""","""75""","""MA""","""6""",0,"""05""","""2024"""


# Filtrar Clientes Ativos para Auditoria

In [692]:
tabela_auditoria = tabela_dados_unicos.filter(pl.col('indcliente') == 1)
tabela_auditoria

id,cpf,nome,sobrenome,idgenero,estado_civil,idade,uf,escolaridade,indcliente,mes,ano
i64,str,str,str,str,str,str,str,str,i64,str,str
2,"""23456700112""","""ANA""","""AMARAL""","""F""","""2""","""56""","""CE""","""6""",1,"""05""","""2024"""
3,"""34567800223""","""ALINE""","""ABREU""","""F""","""1""","""38""","""PE""","""3""",1,"""05""","""2024"""
4,"""45678900334""","""AMANDA""","""ANDRADE""","""F""","""6""","""72""","""BA""","""7""",1,"""05""","""2024"""
5,"""56789000445""","""ADRIANA""","""ARAÚJO""","""F""","""4""","""29""","""MA""","""5""",1,"""05""","""2024"""
6,"""67890100556""","""ALESSANDRA""","""AGUIAR""","""B""","""5""","""45""","""RN""","""5""",1,"""05""","""2024"""
…,…,…,…,…,…,…,…,…,…,…,…
393,"""56198439294""","""CIRO""","""RIOS""","""M""","""1""","""48""","""SE""","""7""",1,"""05""","""2024"""
394,"""68972339395""","""CARLOS EDUARDO""","""RESENDE""","""M""","""2""","""38""","""PE""","""6""",1,"""05""","""2024"""
395,"""37462139496""","""CHARLLES""","""RAMOS""","""M""","""3""","""33""","""CE""","""7""",1,"""05""","""2024"""


# Filtrando Tabela Auditoria

In [693]:
tabela_auditoria = tabela_auditoria.select([
    'id',
    'cpf',
    'nome',
    'sobrenome',
    'idgenero',
    'indcliente',
    'mes',
    'ano'
])
tabela_auditoria

id,cpf,nome,sobrenome,idgenero,indcliente,mes,ano
i64,str,str,str,str,i64,str,str
2,"""23456700112""","""ANA""","""AMARAL""","""F""",1,"""05""","""2024"""
3,"""34567800223""","""ALINE""","""ABREU""","""F""",1,"""05""","""2024"""
4,"""45678900334""","""AMANDA""","""ANDRADE""","""F""",1,"""05""","""2024"""
5,"""56789000445""","""ADRIANA""","""ARAÚJO""","""F""",1,"""05""","""2024"""
6,"""67890100556""","""ALESSANDRA""","""AGUIAR""","""B""",1,"""05""","""2024"""
…,…,…,…,…,…,…,…
393,"""56198439294""","""CIRO""","""RIOS""","""M""",1,"""05""","""2024"""
394,"""68972339395""","""CARLOS EDUARDO""","""RESENDE""","""M""",1,"""05""","""2024"""
395,"""37462139496""","""CHARLLES""","""RAMOS""","""M""",1,"""05""","""2024"""


# Tratamento de Novas Colunas

In [694]:
tabela_auditoria = tabela_auditoria.with_columns(
    pl.concat_str(
        pl.col('nome'),
        pl.lit(' '),
        pl.col('sobrenome')
    ).alias('nome_completo')
)

In [695]:
tabela_auditoria = tabela_auditoria.with_columns(
    pl.concat_str(
        pl.col('ano'),
        pl.col('mes')
    ).cast(int).alias('ano_mes')
)

In [696]:
tabela_auditoria = tabela_auditoria.select(
    [
        'id',
        'cpf',
        'nome_completo',
        'idgenero',
        'indcliente',
        'ano_mes'
    ]
)
tabela_auditoria

id,cpf,nome_completo,idgenero,indcliente,ano_mes
i64,str,str,str,i64,i64
2,"""23456700112""","""ANA AMARAL""","""F""",1,202405
3,"""34567800223""","""ALINE ABREU""","""F""",1,202405
4,"""45678900334""","""AMANDA ANDRADE""","""F""",1,202405
5,"""56789000445""","""ADRIANA ARAÚJO""","""F""",1,202405
6,"""67890100556""","""ALESSANDRA AGUIAR""","""B""",1,202405
…,…,…,…,…,…
393,"""56198439294""","""CIRO RIOS""","""M""",1,202405
394,"""68972339395""","""CARLOS EDUARDO RESENDE""","""M""",1,202405
395,"""37462139496""","""CHARLLES RAMOS""","""M""",1,202405


In [697]:
tabela_auditoria = tabela_auditoria.with_columns(pl.lit(datetime.now()).alias('data_insercao'))
tabela_auditoria

id,cpf,nome_completo,idgenero,indcliente,ano_mes,data_insercao
i64,str,str,str,i64,i64,datetime[μs]
2,"""23456700112""","""ANA AMARAL""","""F""",1,202405,2026-09-14 15:50:13.633285
3,"""34567800223""","""ALINE ABREU""","""F""",1,202405,2026-09-14 15:50:13.633285
4,"""45678900334""","""AMANDA ANDRADE""","""F""",1,202405,2026-09-14 15:50:13.633285
5,"""56789000445""","""ADRIANA ARAÚJO""","""F""",1,202405,2026-09-14 15:50:13.633285
6,"""67890100556""","""ALESSANDRA AGUIAR""","""B""",1,202405,2026-09-14 15:50:13.633285
…,…,…,…,…,…,…
393,"""56198439294""","""CIRO RIOS""","""M""",1,202405,2026-09-14 15:50:13.633285
394,"""68972339395""","""CARLOS EDUARDO RESENDE""","""M""",1,202405,2026-09-14 15:50:13.633285
395,"""37462139496""","""CHARLLES RAMOS""","""M""",1,202405,2026-09-14 15:50:13.633285


In [698]:
BancoDados().inserir_dados(tabela_dados=tabela_auditoria)

# Tratativa Dados Duplicatos

In [699]:
tabela_duplicatos = tabela.filter(pl.col('cpf').is_in(lista_cpfs)).sort(by='id', descending=True).unique(subset=['cpf'])
tabela_duplicatos = tabela_duplicatos.select(
    [
        'id',
        'mes',
        'ano',
        'cpf',
        'nome',
        'sobrenome'
    ]
)
tabela_duplicatos

id,mes,ano,cpf,nome,sobrenome
i64,str,str,str,str,str
504,"""05""","""2024""","""46178257774""","""SOLANO""","""BASTOS"""
506,"""05""","""2024""","""12345678901""","""ALICE""","""ALVES"""
503,"""05""","""2024""","""45656719989""","""ÁGATA""","""HAUGEN"""
501,"""05""","""2024""","""45656709989""","""BÁRBARA""","""TABORDA"""
502,"""05""","""2024""","""34567800223""","""ALINE""","""ABREU"""
507,"""05""","""2024""","""45656729989""","""MÁRCIA""","""TAVARES"""


In [700]:
CAMINHO_DESTINO: str = r"C:\Users\JoãoPedroResplandesd\Estudos\Pos_Graduacao_Engenharia_de_Dados\Preparacao_Orquestacao_Fluxo_de_Dados\Unidade02\Materiais\ProjetoChurn\Destino"

In [701]:
tabela_duplicatos.write_excel(rf'{CAMINHO_DESTINO}\CPFS_DUPLICADOS.xlsx', worksheet='Duplicados')

# ETL -> MKT

In [702]:
tabela_dados_unicos

id,cpf,nome,sobrenome,idgenero,estado_civil,idade,uf,escolaridade,indcliente,mes,ano
i64,str,str,str,str,str,str,str,str,i64,str,str
1,"""12345678901""","""ALICE""","""ALVES""","""F""","""3""","""25""","""SE""","""11""",0,"""05""","""2024"""
2,"""23456700112""","""ANA""","""AMARAL""","""F""","""2""","""56""","""CE""","""6""",1,"""05""","""2024"""
3,"""34567800223""","""ALINE""","""ABREU""","""F""","""1""","""38""","""PE""","""3""",1,"""05""","""2024"""
4,"""45678900334""","""AMANDA""","""ANDRADE""","""F""","""6""","""72""","""BA""","""7""",1,"""05""","""2024"""
5,"""56789000445""","""ADRIANA""","""ARAÚJO""","""F""","""4""","""29""","""MA""","""5""",1,"""05""","""2024"""
…,…,…,…,…,…,…,…,…,…,…,…
496,"""57293457370""","""SIMÃO""","""BERNARDES""","""M""","""2""","""46""","""PB""","""6""",0,"""05""","""2024"""
497,"""95417857471""","""SÉRGIO""","""BORGES""","""M""","""3""","""56""","""CE""","""7""",0,"""05""","""2024"""
498,"""81942657572""","""SALOMÃO""","""BITTENCOURT""","""M""","""4""","""75""","""MA""","""6""",0,"""05""","""2024"""


In [703]:
tabela_mkt = tabela_dados_unicos.filter(pl.col('indcliente') != 1)
tabela_mkt = tabela_mkt.with_columns(
    pl.concat_str(
        pl.col('ano'),
        pl.col('mes')
    ).cast(int).alias('ano_mes')
)
tabela_mkt = tabela_mkt.with_columns(pl.col('estado_civil').cast(int).alias('codestadocivil'))
tabela_mkt = tabela_mkt.with_columns(pl.col('idade').cast(int).alias('idade'))
tabela_mkt = tabela_mkt.with_columns(pl.col('escolaridade').cast(int).alias('codescolaridade'))
tabela_mkt

id,cpf,nome,sobrenome,idgenero,estado_civil,idade,uf,escolaridade,indcliente,mes,ano,ano_mes,codestadocivil,codescolaridade
i64,str,str,str,str,str,i64,str,str,i64,str,str,i64,i64,i64
1,"""12345678901""","""ALICE""","""ALVES""","""F""","""3""",25,"""SE""","""11""",0,"""05""","""2024""",202405,3,11
8,"""89012300778""","""ALÍCIA""","""AZEVEDO""","""F""","""2""",34,"""PI""","""6""",0,"""05""","""2024""",202405,2,6
9,"""90123400889""","""ANDREIA""","""AMORIM""","""F""","""6""",81,"""PB""","""1""",0,"""05""","""2024""",202405,6,1
10,"""01234500990""","""ANTÔNIA""","""ALENCAR""","""F""","""2""",44,"""CE""","""6""",0,"""05""","""2024""",202405,2,6
11,"""09876501010""","""AYLA""","""ANTUNES""","""F""","""4""",43,"""RN""","""9""",0,"""05""","""2024""",202405,4,9
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
496,"""57293457370""","""SIMÃO""","""BERNARDES""","""M""","""2""",46,"""PB""","""6""",0,"""05""","""2024""",202405,2,6
497,"""95417857471""","""SÉRGIO""","""BORGES""","""M""","""3""",56,"""CE""","""7""",0,"""05""","""2024""",202405,3,7
498,"""81942657572""","""SALOMÃO""","""BITTENCOURT""","""M""","""4""",75,"""MA""","""6""",0,"""05""","""2024""",202405,4,6


In [704]:
tabela_mkt = tabela_mkt.select(
    [
        'id',
        'idgenero',
        'codestadocivil',
        'idade',
        'uf',
        'codescolaridade',
        'indcliente',
        'ano_mes'
    ]
)
tabela_mkt

id,idgenero,codestadocivil,idade,uf,codescolaridade,indcliente,ano_mes
i64,str,i64,i64,str,i64,i64,i64
1,"""F""",3,25,"""SE""",11,0,202405
8,"""F""",2,34,"""PI""",6,0,202405
9,"""F""",6,81,"""PB""",1,0,202405
10,"""F""",2,44,"""CE""",6,0,202405
11,"""F""",4,43,"""RN""",9,0,202405
…,…,…,…,…,…,…,…
496,"""M""",2,46,"""PB""",6,0,202405
497,"""M""",3,56,"""CE""",7,0,202405
498,"""M""",4,75,"""MA""",6,0,202405


In [705]:
tabela_mkt.write_csv(Rf'{CAMINHO_DESTINO}\1-CHURN_PERFIL.csv', separator=';')

# Enriquecimento de Dados

In [706]:
ARQUIVO_ESTADO_CIVIL: str = r"C:\Users\JoãoPedroResplandesd\Estudos\Pos_Graduacao_Engenharia_de_Dados\Preparacao_Orquestacao_Fluxo_de_Dados\Unidade03\Materiais\estadocivil.json"
ARQUIVO_ESCOLARIDADE: str = r"C:\Users\JoãoPedroResplandesd\Estudos\Pos_Graduacao_Engenharia_de_Dados\Preparacao_Orquestacao_Fluxo_de_Dados\Unidade03\Materiais\escolaridade.txt"

## Estado Civil

In [707]:
with open(ARQUIVO_ESTADO_CIVIL, 'r', encoding='utf-8') as arquivo_json:
    dados_estado_civil = json.loads(arquivo_json.read())

In [708]:
dict_estado_civil = {
    'ID_EC': [],
    'DSC_EC': []
}
for item in dados_estado_civil:
    dict_estado_civil['ID_EC'].append(item['ID_EC'])
    dict_estado_civil['DSC_EC'].append(item['DSC_EC'])

tb_estado_civil = pl.DataFrame(dict_estado_civil)
tb_estado_civil

ID_EC,DSC_EC
i64,str
1,"""Solteiro(a)"""
2,"""Casado(a)"""
3,"""Divorciado(a)"""
4,"""Viúvo(a)"""
5,"""União Estável"""
6,"""Separado(a)"""


In [709]:
tabela_mkt = tabela_mkt.join(tb_estado_civil, left_on='codestadocivil', right_on='ID_EC', how='left')
tabela_mkt

id,idgenero,codestadocivil,idade,uf,codescolaridade,indcliente,ano_mes,DSC_EC
i64,str,i64,i64,str,i64,i64,i64,str
1,"""F""",3,25,"""SE""",11,0,202405,"""Divorciado(a)"""
8,"""F""",2,34,"""PI""",6,0,202405,"""Casado(a)"""
9,"""F""",6,81,"""PB""",1,0,202405,"""Separado(a)"""
10,"""F""",2,44,"""CE""",6,0,202405,"""Casado(a)"""
11,"""F""",4,43,"""RN""",9,0,202405,"""Viúvo(a)"""
…,…,…,…,…,…,…,…,…
496,"""M""",2,46,"""PB""",6,0,202405,"""Casado(a)"""
497,"""M""",3,56,"""CE""",7,0,202405,"""Divorciado(a)"""
498,"""M""",4,75,"""MA""",6,0,202405,"""Viúvo(a)"""


## Escolaridade

In [710]:
with open(ARQUIVO_ESCOLARIDADE, 'r', encoding='utf-8') as arquivo_txt:
    dados_escolaridade = arquivo_txt.readlines()

In [711]:
lista_dados_escolaridade = [dado.replace('\n', '').split(';') for dado in dados_escolaridade]
lista_dados_escolaridade

[['ID_ESCOLARIDADE', 'DESC_ESCOLARIDADE'],
 ['1', 'Ensino Fundamental Incompleto'],
 ['2', 'Ensino Fundamental Completo'],
 ['3', 'Ensino Médio Incompleto'],
 ['4', 'Ensino Médio Completo'],
 ['5', 'Ensino Superior Incompleto'],
 ['6', 'Bacharelado'],
 ['7', 'Mestrado'],
 ['8', 'Doutorado'],
 ['9', 'Pós-Graduação']]

In [712]:
dict_escolaridade = {}
for posicao, coluna in enumerate(lista_dados_escolaridade[0]):
    lista_dados = []
    for linha in lista_dados_escolaridade[1:]:
        lista_dados.append(linha[posicao])
    dict_escolaridade[coluna] = lista_dados

tb_escolaridade = pl.DataFrame(dict_escolaridade)
tb_escolaridade = tb_escolaridade.with_columns(pl.col('ID_ESCOLARIDADE').cast(int).alias('ID_ESCOLARIDADE'))
tb_escolaridade

ID_ESCOLARIDADE,DESC_ESCOLARIDADE
i64,str
1,"""Ensino Fundamental Incompleto"""
2,"""Ensino Fundamental Completo"""
3,"""Ensino Médio Incompleto"""
4,"""Ensino Médio Completo"""
5,"""Ensino Superior Incompleto"""
6,"""Bacharelado"""
7,"""Mestrado"""
8,"""Doutorado"""
9,"""Pós-Graduação"""


In [713]:
tabela_mkt = tabela_mkt.join(
    tb_escolaridade,
    left_on='codescolaridade',
    right_on='ID_ESCOLARIDADE',
    how='left'
)

tabela_mkt

id,idgenero,codestadocivil,idade,uf,codescolaridade,indcliente,ano_mes,DSC_EC,DESC_ESCOLARIDADE
i64,str,i64,i64,str,i64,i64,i64,str,str
1,"""F""",3,25,"""SE""",11,0,202405,"""Divorciado(a)""",null
8,"""F""",2,34,"""PI""",6,0,202405,"""Casado(a)""","""Bacharelado"""
9,"""F""",6,81,"""PB""",1,0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto"""
10,"""F""",2,44,"""CE""",6,0,202405,"""Casado(a)""","""Bacharelado"""
11,"""F""",4,43,"""RN""",9,0,202405,"""Viúvo(a)""","""Pós-Graduação"""
…,…,…,…,…,…,…,…,…,…
496,"""M""",2,46,"""PB""",6,0,202405,"""Casado(a)""","""Bacharelado"""
497,"""M""",3,56,"""CE""",7,0,202405,"""Divorciado(a)""","""Mestrado"""
498,"""M""",4,75,"""MA""",6,0,202405,"""Viúvo(a)""","""Bacharelado"""


## Consulta Perfil Banco

In [714]:
def tratar_valores(valor: float):
    return int(round(valor, 0))

In [715]:
tabela_perfil = BancoDados().consulta_perfil_cliente(lista_ids=tuple(tabela_mkt['id'].unique().to_list()))
tabela_perfil

ID,VLR_MENSAL,VLR_COPARTICIPACAO
i64,"decimal[38,2]","decimal[38,2]"
1,550.00,3725.00
8,750.00,3910.00
9,1850.00,168.00
10,697.00,7746.00
11,875.00,3154.00
…,…,…
496,697.00,5048.00
497,1035.00,5123.00
498,1500.00,5617.00


In [716]:
tabela_mkt = tabela_mkt.join(
    tabela_perfil,
    left_on='id',
    right_on='ID',
    how='left'
)
tabela_mkt = tabela_mkt.with_columns(pl.col('VLR_MENSAL').map_elements(tratar_valores).alias('VLR_MENSAL'))
tabela_mkt = tabela_mkt.with_columns(pl.col('VLR_COPARTICIPACAO').map_elements(tratar_valores).alias('VLR_COPARTICIPACAO'))
tabela_mkt

id,idgenero,codestadocivil,idade,uf,codescolaridade,indcliente,ano_mes,DSC_EC,DESC_ESCOLARIDADE,VLR_MENSAL,VLR_COPARTICIPACAO
i64,str,i64,i64,str,i64,i64,i64,str,str,i64,i64
1,"""F""",3,25,"""SE""",11,0,202405,"""Divorciado(a)""",null,550,3725
8,"""F""",2,34,"""PI""",6,0,202405,"""Casado(a)""","""Bacharelado""",750,3910
9,"""F""",6,81,"""PB""",1,0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto""",1850,168
10,"""F""",2,44,"""CE""",6,0,202405,"""Casado(a)""","""Bacharelado""",697,7746
11,"""F""",4,43,"""RN""",9,0,202405,"""Viúvo(a)""","""Pós-Graduação""",875,3154
…,…,…,…,…,…,…,…,…,…,…,…
496,"""M""",2,46,"""PB""",6,0,202405,"""Casado(a)""","""Bacharelado""",697,5048
497,"""M""",3,56,"""CE""",7,0,202405,"""Divorciado(a)""","""Mestrado""",1035,5123
498,"""M""",4,75,"""MA""",6,0,202405,"""Viúvo(a)""","""Bacharelado""",1500,5617


## Carga Arquivo

In [717]:
tabela_mkt = tabela_mkt.rename(
    {
        'idgenero':'genero',
        'DSC_EC':'estado_civil',
        'DESC_ESCOLARIDADE':'escolaridade',
        'VLR_MENSAL':'mensalidade',
        'VLR_COPARTICIPACAO':'participacao'
    }
)

tabela_mkt

id,genero,codestadocivil,idade,uf,codescolaridade,indcliente,ano_mes,estado_civil,escolaridade,mensalidade,participacao
i64,str,i64,i64,str,i64,i64,i64,str,str,i64,i64
1,"""F""",3,25,"""SE""",11,0,202405,"""Divorciado(a)""",null,550,3725
8,"""F""",2,34,"""PI""",6,0,202405,"""Casado(a)""","""Bacharelado""",750,3910
9,"""F""",6,81,"""PB""",1,0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto""",1850,168
10,"""F""",2,44,"""CE""",6,0,202405,"""Casado(a)""","""Bacharelado""",697,7746
11,"""F""",4,43,"""RN""",9,0,202405,"""Viúvo(a)""","""Pós-Graduação""",875,3154
…,…,…,…,…,…,…,…,…,…,…,…
496,"""M""",2,46,"""PB""",6,0,202405,"""Casado(a)""","""Bacharelado""",697,5048
497,"""M""",3,56,"""CE""",7,0,202405,"""Divorciado(a)""","""Mestrado""",1035,5123
498,"""M""",4,75,"""MA""",6,0,202405,"""Viúvo(a)""","""Bacharelado""",1500,5617


In [718]:
tabela_mkt = tabela_mkt.select(
    [
        'id',
        'genero',
        'idade',
        'uf',
        'indcliente',
        'ano_mes',
        'estado_civil',
        'escolaridade',
        'mensalidade',
        'participacao'
    ]
)

tabela_mkt

id,genero,idade,uf,indcliente,ano_mes,estado_civil,escolaridade,mensalidade,participacao
i64,str,i64,str,i64,i64,str,str,i64,i64
1,"""F""",25,"""SE""",0,202405,"""Divorciado(a)""",null,550,3725
8,"""F""",34,"""PI""",0,202405,"""Casado(a)""","""Bacharelado""",750,3910
9,"""F""",81,"""PB""",0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto""",1850,168
10,"""F""",44,"""CE""",0,202405,"""Casado(a)""","""Bacharelado""",697,7746
11,"""F""",43,"""RN""",0,202405,"""Viúvo(a)""","""Pós-Graduação""",875,3154
…,…,…,…,…,…,…,…,…,…
496,"""M""",46,"""PB""",0,202405,"""Casado(a)""","""Bacharelado""",697,5048
497,"""M""",56,"""CE""",0,202405,"""Divorciado(a)""","""Mestrado""",1035,5123
498,"""M""",75,"""MA""",0,202405,"""Viúvo(a)""","""Bacharelado""",1500,5617


In [719]:
# tabela_mkt.write_csv(rf'{CAMINHO_DESTINO}\2-CHURN_PERFIL_ENRIQUECIDO.csv', separator=';')

# Validação

### Tratando Idade

In [720]:
tabela_mkt = tabela_mkt.with_columns(
    pl.when(
        (pl.col('idade') < 0)
        |
        (pl.col('idade') > 100)
    ).then(
        pl.lit(ceil(tabela_mkt['idade'].median()))
    ).otherwise(pl.col('idade')).alias('idade')
)

tabela_mkt

id,genero,idade,uf,indcliente,ano_mes,estado_civil,escolaridade,mensalidade,participacao
i64,str,i64,str,i64,i64,str,str,i64,i64
1,"""F""",25,"""SE""",0,202405,"""Divorciado(a)""",null,550,3725
8,"""F""",34,"""PI""",0,202405,"""Casado(a)""","""Bacharelado""",750,3910
9,"""F""",81,"""PB""",0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto""",1850,168
10,"""F""",44,"""CE""",0,202405,"""Casado(a)""","""Bacharelado""",697,7746
11,"""F""",43,"""RN""",0,202405,"""Viúvo(a)""","""Pós-Graduação""",875,3154
…,…,…,…,…,…,…,…,…,…
496,"""M""",46,"""PB""",0,202405,"""Casado(a)""","""Bacharelado""",697,5048
497,"""M""",56,"""CE""",0,202405,"""Divorciado(a)""","""Mestrado""",1035,5123
498,"""M""",75,"""MA""",0,202405,"""Viúvo(a)""","""Bacharelado""",1500,5617


### Tratando UF

In [721]:
response = requests.get(f'https://servicodados.ibge.gov.br/api/v1/localidades/estados')
resposta = response.json()

dicionario_regiao_nordeste = {}

for item in resposta:
    if item['regiao']['nome'] == 'Nordeste':
        dicionario_regiao_nordeste[item['sigla']] = {'estado': item['nome'], 'regiao': item['regiao']['nome']}

dicionario_regiao_nordeste

{'MA': {'estado': 'Maranhão', 'regiao': 'Nordeste'},
 'PI': {'estado': 'Piauí', 'regiao': 'Nordeste'},
 'CE': {'estado': 'Ceará', 'regiao': 'Nordeste'},
 'RN': {'estado': 'Rio Grande do Norte', 'regiao': 'Nordeste'},
 'PB': {'estado': 'Paraíba', 'regiao': 'Nordeste'},
 'PE': {'estado': 'Pernambuco', 'regiao': 'Nordeste'},
 'AL': {'estado': 'Alagoas', 'regiao': 'Nordeste'},
 'SE': {'estado': 'Sergipe', 'regiao': 'Nordeste'},
 'BA': {'estado': 'Bahia', 'regiao': 'Nordeste'}}

Substituindo os UFS que não pertencem ao Nordeste pela a UF que contem o maior registro

In [722]:
tabela_mkt = tabela_mkt.with_columns(
    pl.when(
        ~pl.col('uf').is_in(dicionario_regiao_nordeste)
    ).then(
        pl.lit(tabela_mkt['uf'].value_counts().sort(by='count', descending=True)[0]['uf'].to_list()[0])
    ).otherwise(pl.col('uf'))
    .alias('uf')
    )

tabela_mkt

id,genero,idade,uf,indcliente,ano_mes,estado_civil,escolaridade,mensalidade,participacao
i64,str,i64,str,i64,i64,str,str,i64,i64
1,"""F""",25,"""SE""",0,202405,"""Divorciado(a)""",null,550,3725
8,"""F""",34,"""PI""",0,202405,"""Casado(a)""","""Bacharelado""",750,3910
9,"""F""",81,"""PB""",0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto""",1850,168
10,"""F""",44,"""CE""",0,202405,"""Casado(a)""","""Bacharelado""",697,7746
11,"""F""",43,"""RN""",0,202405,"""Viúvo(a)""","""Pós-Graduação""",875,3154
…,…,…,…,…,…,…,…,…,…
496,"""M""",46,"""PB""",0,202405,"""Casado(a)""","""Bacharelado""",697,5048
497,"""M""",56,"""CE""",0,202405,"""Divorciado(a)""","""Mestrado""",1035,5123
498,"""M""",75,"""MA""",0,202405,"""Viúvo(a)""","""Bacharelado""",1500,5617


### Tratar estado civil

In [723]:
tabela_mkt = tabela_mkt.with_columns(
    pl.when(
        ~pl.col('estado_civil').is_in(tb_estado_civil['DSC_EC'].unique().to_list())
        |
        pl.col('estado_civil').is_null()
    ).then(
        pl.lit(tabela_mkt['estado_civil'].value_counts().sort(by='count', descending=True)[0]['estado_civil'].to_list()[0])
    ).otherwise(pl.col('estado_civil'))
    .alias('estado_civil')
)

tabela_mkt

id,genero,idade,uf,indcliente,ano_mes,estado_civil,escolaridade,mensalidade,participacao
i64,str,i64,str,i64,i64,str,str,i64,i64
1,"""F""",25,"""SE""",0,202405,"""Divorciado(a)""",null,550,3725
8,"""F""",34,"""PI""",0,202405,"""Casado(a)""","""Bacharelado""",750,3910
9,"""F""",81,"""PB""",0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto""",1850,168
10,"""F""",44,"""CE""",0,202405,"""Casado(a)""","""Bacharelado""",697,7746
11,"""F""",43,"""RN""",0,202405,"""Viúvo(a)""","""Pós-Graduação""",875,3154
…,…,…,…,…,…,…,…,…,…
496,"""M""",46,"""PB""",0,202405,"""Casado(a)""","""Bacharelado""",697,5048
497,"""M""",56,"""CE""",0,202405,"""Divorciado(a)""","""Mestrado""",1035,5123
498,"""M""",75,"""MA""",0,202405,"""Viúvo(a)""","""Bacharelado""",1500,5617


### Tratar Escolaridade

In [724]:
tabela_mkt = tabela_mkt.with_columns(
    pl.when(
        ~pl.col('escolaridade').is_in([escolaridade for escolaridade in tb_escolaridade['DESC_ESCOLARIDADE'].unique().to_list()])
        |
        pl.col('escolaridade').is_null()
    ).then(
        pl.lit(
            tabela_mkt['escolaridade'].value_counts().sort(by='count', descending=True)[0]['escolaridade'].to_list()[0])
        )
    .otherwise(pl.col('escolaridade'))
    .alias('escolaridade')
)

tabela_mkt

id,genero,idade,uf,indcliente,ano_mes,estado_civil,escolaridade,mensalidade,participacao
i64,str,i64,str,i64,i64,str,str,i64,i64
1,"""F""",25,"""SE""",0,202405,"""Divorciado(a)""","""Bacharelado""",550,3725
8,"""F""",34,"""PI""",0,202405,"""Casado(a)""","""Bacharelado""",750,3910
9,"""F""",81,"""PB""",0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto""",1850,168
10,"""F""",44,"""CE""",0,202405,"""Casado(a)""","""Bacharelado""",697,7746
11,"""F""",43,"""RN""",0,202405,"""Viúvo(a)""","""Pós-Graduação""",875,3154
…,…,…,…,…,…,…,…,…,…
496,"""M""",46,"""PB""",0,202405,"""Casado(a)""","""Bacharelado""",697,5048
497,"""M""",56,"""CE""",0,202405,"""Divorciado(a)""","""Mestrado""",1035,5123
498,"""M""",75,"""MA""",0,202405,"""Viúvo(a)""","""Bacharelado""",1500,5617


### Tratar Mensalidade

In [725]:
def mensalidade_por_idade(num_idade: int) -> int:
    agrupamento_idade_mensalidade = tabela_mkt.group_by('idade').agg(
        pl.col('mensalidade').mean().round(0).cast(int).alias('media_mensalidade')
    ).sort(by='idade', descending=True)

    dicionario_agrupamento = {}
    for idade in agrupamento_idade_mensalidade['idade']:
        dicionario_agrupamento[idade] = agrupamento_idade_mensalidade.filter(pl.col('idade') == idade)['media_mensalidade'].to_list()[0]

    return dicionario_agrupamento[num_idade]

In [726]:
tabela_mkt = pl.concat(
    [
        tabela_mkt.filter(~pl.col('mensalidade').is_null()),
        tabela_mkt.filter(pl.col('mensalidade').is_null()).with_columns(
        pl.col('idade')
        .map_elements(mensalidade_por_idade, return_dtype=pl.Int64).alias('mensalidade'))
    ]
)
tabela_mkt

id,genero,idade,uf,indcliente,ano_mes,estado_civil,escolaridade,mensalidade,participacao
i64,str,i64,str,i64,i64,str,str,i64,i64
1,"""F""",25,"""SE""",0,202405,"""Divorciado(a)""","""Bacharelado""",550,3725
8,"""F""",34,"""PI""",0,202405,"""Casado(a)""","""Bacharelado""",750,3910
9,"""F""",81,"""PB""",0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto""",1850,168
10,"""F""",44,"""CE""",0,202405,"""Casado(a)""","""Bacharelado""",697,7746
11,"""F""",43,"""RN""",0,202405,"""Viúvo(a)""","""Pós-Graduação""",875,3154
…,…,…,…,…,…,…,…,…,…
285,"""F""",46,"""RN""",0,202405,"""Casado(a)""","""Bacharelado""",768,2906
288,"""F""",46,"""CE""",0,202405,"""União Estável""","""Ensino Fundamental Completo""",768,2977
295,"""F""",48,"""RN""",0,202405,"""Separado(a)""","""Bacharelado""",816,2192


### Tratar Coparticipação

In [727]:
desvio_padrao_considerado = tabela_mkt['participacao'].std() * 4
desvio_padrao_considerado

9994.796128783688

In [728]:
tabela_mkt = tabela_mkt.with_columns(
    pl.when(
        (pl.col('participacao') >= desvio_padrao_considerado)
        |
        (pl.col('participacao').is_null())
    ).then(pl.lit(ceil(tabela_mkt['participacao'].median())))
    .otherwise(pl.col('participacao'))
    .alias('participacao')
)

tabela_mkt

id,genero,idade,uf,indcliente,ano_mes,estado_civil,escolaridade,mensalidade,participacao
i64,str,i64,str,i64,i64,str,str,i64,i64
1,"""F""",25,"""SE""",0,202405,"""Divorciado(a)""","""Bacharelado""",550,3725
8,"""F""",34,"""PI""",0,202405,"""Casado(a)""","""Bacharelado""",750,3910
9,"""F""",81,"""PB""",0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto""",1850,168
10,"""F""",44,"""CE""",0,202405,"""Casado(a)""","""Bacharelado""",697,7746
11,"""F""",43,"""RN""",0,202405,"""Viúvo(a)""","""Pós-Graduação""",875,3154
…,…,…,…,…,…,…,…,…,…
285,"""F""",46,"""RN""",0,202405,"""Casado(a)""","""Bacharelado""",768,2906
288,"""F""",46,"""CE""",0,202405,"""União Estável""","""Ensino Fundamental Completo""",768,2977
295,"""F""",48,"""RN""",0,202405,"""Separado(a)""","""Bacharelado""",816,2192


### Exportar Base

In [729]:
tabela_mkt.write_csv(rf'{CAMINHO_DESTINO}\2-CHURN_PERFIL_ENRIQUECIDO.csv', separator=';')

# Perfil Analítico

In [730]:
tabela_mkt

id,genero,idade,uf,indcliente,ano_mes,estado_civil,escolaridade,mensalidade,participacao
i64,str,i64,str,i64,i64,str,str,i64,i64
1,"""F""",25,"""SE""",0,202405,"""Divorciado(a)""","""Bacharelado""",550,3725
8,"""F""",34,"""PI""",0,202405,"""Casado(a)""","""Bacharelado""",750,3910
9,"""F""",81,"""PB""",0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto""",1850,168
10,"""F""",44,"""CE""",0,202405,"""Casado(a)""","""Bacharelado""",697,7746
11,"""F""",43,"""RN""",0,202405,"""Viúvo(a)""","""Pós-Graduação""",875,3154
…,…,…,…,…,…,…,…,…,…
285,"""F""",46,"""RN""",0,202405,"""Casado(a)""","""Bacharelado""",768,2906
288,"""F""",46,"""CE""",0,202405,"""União Estável""","""Ensino Fundamental Completo""",768,2977
295,"""F""",48,"""RN""",0,202405,"""Separado(a)""","""Bacharelado""",816,2192


In [731]:
def faixa_etaria(idade: int) -> str:
    if idade > 70:
        return 'Idoso'
    elif idade >= 51 and idade <= 70:
        return 'Adulto Maduro'
    elif idade >= 30 and idade <= 50:
        return 'Adulto Jovem'
    else:
        return 'Jovem'

In [732]:
def grupo_escolaridade(escolaridade: str) -> str:
    if escolaridade in ('Pós-Graduação', 'Doutorado', 'Mestrado'):
        return 'Stricto/Lato Sensu'
    else:
        return escolaridade

In [733]:
faixa_etaria(15)

'Jovem'

#### Faixa Etária

In [734]:
tabela_analitico = tabela_mkt.with_columns(
    pl.col('idade').map_elements(faixa_etaria, return_dtype=pl.Utf8).alias('faixa_etaria')
)
tabela_analitico

id,genero,idade,uf,indcliente,ano_mes,estado_civil,escolaridade,mensalidade,participacao,faixa_etaria
i64,str,i64,str,i64,i64,str,str,i64,i64,str
1,"""F""",25,"""SE""",0,202405,"""Divorciado(a)""","""Bacharelado""",550,3725,"""Jovem"""
8,"""F""",34,"""PI""",0,202405,"""Casado(a)""","""Bacharelado""",750,3910,"""Adulto Jovem"""
9,"""F""",81,"""PB""",0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto""",1850,168,"""Idoso"""
10,"""F""",44,"""CE""",0,202405,"""Casado(a)""","""Bacharelado""",697,7746,"""Adulto Jovem"""
11,"""F""",43,"""RN""",0,202405,"""Viúvo(a)""","""Pós-Graduação""",875,3154,"""Adulto Jovem"""
…,…,…,…,…,…,…,…,…,…,…
285,"""F""",46,"""RN""",0,202405,"""Casado(a)""","""Bacharelado""",768,2906,"""Adulto Jovem"""
288,"""F""",46,"""CE""",0,202405,"""União Estável""","""Ensino Fundamental Completo""",768,2977,"""Adulto Jovem"""
295,"""F""",48,"""RN""",0,202405,"""Separado(a)""","""Bacharelado""",816,2192,"""Adulto Jovem"""


#### Escolaridade

In [735]:
tabela_analitico = tabela_analitico.with_columns(
    pl.col('escolaridade').map_elements(grupo_escolaridade, return_dtype=pl.Utf8).alias('escolaridade')
)
tabela_analitico

id,genero,idade,uf,indcliente,ano_mes,estado_civil,escolaridade,mensalidade,participacao,faixa_etaria
i64,str,i64,str,i64,i64,str,str,i64,i64,str
1,"""F""",25,"""SE""",0,202405,"""Divorciado(a)""","""Bacharelado""",550,3725,"""Jovem"""
8,"""F""",34,"""PI""",0,202405,"""Casado(a)""","""Bacharelado""",750,3910,"""Adulto Jovem"""
9,"""F""",81,"""PB""",0,202405,"""Separado(a)""","""Ensino Fundamental Incompleto""",1850,168,"""Idoso"""
10,"""F""",44,"""CE""",0,202405,"""Casado(a)""","""Bacharelado""",697,7746,"""Adulto Jovem"""
11,"""F""",43,"""RN""",0,202405,"""Viúvo(a)""","""Stricto/Lato Sensu""",875,3154,"""Adulto Jovem"""
…,…,…,…,…,…,…,…,…,…,…
285,"""F""",46,"""RN""",0,202405,"""Casado(a)""","""Bacharelado""",768,2906,"""Adulto Jovem"""
288,"""F""",46,"""CE""",0,202405,"""União Estável""","""Ensino Fundamental Completo""",768,2977,"""Adulto Jovem"""
295,"""F""",48,"""RN""",0,202405,"""Separado(a)""","""Bacharelado""",816,2192,"""Adulto Jovem"""


#### Agrupamento

In [736]:
tabela_analitico = tabela_analitico.group_by(['genero', 'uf', 'ano_mes', 'estado_civil', 'escolaridade', 'faixa_etaria']).agg(
    pl.col('id').count().alias('total_churn'),
    pl.col('mensalidade').sum().alias('valor_mensalidade_mes'),
    pl.col('participacao').sum().alias('valor_participacao_mes')
)

tabela_analitico

genero,uf,ano_mes,estado_civil,escolaridade,faixa_etaria,total_churn,valor_mensalidade_mes,valor_participacao_mes
str,str,i64,str,str,str,u32,i64,i64
"""M""","""PE""",202405,"""Casado(a)""","""Stricto/Lato Sensu""","""Adulto Jovem""",2,1447,4811
"""M""","""MA""",202405,"""União Estável""","""Bacharelado""","""Adulto Jovem""",1,697,4047
"""M""","""RN""",202405,"""Solteiro(a)""","""Bacharelado""","""Adulto Maduro""",2,2345,7666
"""F""","""MA""",202405,"""Solteiro(a)""","""Ensino Superior Incompleto""","""Adulto Jovem""",1,768,7998
"""F""","""CE""",202405,"""Separado(a)""","""Stricto/Lato Sensu""","""Idoso""",1,1850,4062
…,…,…,…,…,…,…,…,…
"""M""","""RN""",202405,"""Viúvo(a)""","""Bacharelado""","""Adulto Jovem""",1,750,987
"""F""","""MA""",202405,"""Casado(a)""","""Bacharelado""","""Adulto Jovem""",1,675,5076
"""F""","""SE""",202405,"""Divorciado(a)""","""Ensino Superior Incompleto""","""Idoso""",1,1500,3702


In [737]:
tabela_analitico.write_csv(rf'{CAMINHO_DESTINO}\3-CHURN_PERFIL_ANALITICO.csv', separator=';')